# TATTLETALE — Kaggle T4/P100 Run

Full pipeline: LoRA fine-tune → extract diff vectors → steer + evaluate.

**Setup:** Upload the `tattletale/` repo as a Kaggle dataset, or clone from GitHub.
Enable GPU accelerator (T4 or P100).

In [ ]:
# 0. Install dependencies
!pip install -q transformers>=4.46 peft>=0.13 bitsandbytes>=0.44 accelerate>=1.0 tqdm

In [ ]:
# 0b. Set paths — adjust if using Kaggle dataset input
import os, sys

# If running from the repo root:
PROJECT_ROOT = os.path.abspath('.')
# If the repo is a Kaggle dataset input, uncomment:
# PROJECT_ROOT = '/kaggle/input/tattletale'

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'GPU: {os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip()}')

## Step 1: Fine-tune LoRA adapters (3 behaviors)

~15-20 min per behavior on T4. Total ~45-60 min.

In [ ]:
# 1a. Train gaming adapter
!python src/train_lora.py --behavior gaming --epochs 3 --batch_size 4 --grad_accum 4

In [ ]:
# 1b. Train verbose adapter
!python src/train_lora.py --behavior verbose --epochs 3 --batch_size 4 --grad_accum 4

In [ ]:
# 1c. Train clean adapter (control)
!python src/train_lora.py --behavior clean --epochs 3 --batch_size 4 --grad_accum 4

In [ ]:
# Verify adapters exist
import os
for b in ('gaming', 'verbose', 'clean'):
    path = f'adapters/{b}'
    files = os.listdir(path) if os.path.exists(path) else []
    print(f'{b}: {len(files)} files — {files[:5]}')

## Step 2: Extract diff-of-means vectors

Runs eval prompts through base + each fine-tuned model, computes diff vectors.
~10 min.

In [ ]:
!python src/extract_vectors.py --layer 15

In [ ]:
# Check cosine similarities — key diagnostic
import json
with open('results/extraction_meta.json') as f:
    meta = json.load(f)
print('Cosine similarities:')
for k, v in meta['cosine_similarities'].items():
    print(f'  {k}: {v:.4f}')
print()
print('If gaming_vs_verbose < 0.9, vectors encode behavior, not just topic.')
print('If gaming_vs_verbose > 0.9, topic dominates — H0 holds.')

## Step 3: Steer + evaluate

Add each diff vector to the base model at inference time, evaluate on held-out problems.
Alpha sweep to find optimal steering strength.

In [ ]:
# Alpha sweep — test multiple strengths
!python src/steer_eval.py --layer 15 --alphas 2,4,6,8 --max_problems 50

In [ ]:
# Full eval at best alpha (update alpha value based on sweep above)
!python src/steer_eval.py --layer 15 --alpha 4.0

## Step 4: Summary metrics + gate verdicts

In [ ]:
!python src/metrics.py

## Step 5: Save results

Download results/ folder for the write-up.

In [ ]:
# List all result files
import os
for f in sorted(os.listdir('results')):
    size = os.path.getsize(f'results/{f}')
    print(f'  results/{f}  ({size:,} bytes)')

In [ ]:
# Archive for download
!tar czf /kaggle/working/tattletale_results.tar.gz results/ adapters/